In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 71.3 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have

In [ ]:
!pip install -U --no-cache-dir numpy==1.26.4 scipy==1.11.4 scikit-learn==1.3.2


In [2]:
import os
import numpy as np
import shutil
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Paths
dataset_path = "/kaggle/input/datasets/umangjpatel/ham10000-imagenet-style-dataset"
base_dir = "/kaggle/working/split_data"

# Selected ISIC classes
selected_classes = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]

# Create split directories
for split in ['train', 'val', 'test']:
    for class_name in selected_classes:
        os.makedirs(os.path.join(base_dir, split, class_name), exist_ok=True)

# Split data
for class_name in selected_classes:
    class_path = os.path.join(dataset_path, class_name)
    
    if not os.path.exists(class_path):
        print(f"Skipping missing class: {class_name}")
        continue

    images = os.listdir(class_path)

    train_files, temp_files = train_test_split(
        images, test_size=0.3, random_state=42
    )
    val_files, test_files = train_test_split(
        temp_files, test_size=0.5, random_state=42
    )

    for file in train_files:
        shutil.copy(
            os.path.join(class_path, file),
            os.path.join(base_dir, 'train', class_name, file)
        )

    for file in val_files:
        shutil.copy(
            os.path.join(class_path, file),
            os.path.join(base_dir, 'val', class_name, file)
        )

    for file in test_files:
        shutil.copy(
            os.path.join(class_path, file),
            os.path.join(base_dir, 'test', class_name, file)
        )

# Data generators
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=(224, 224),  # recommended for MobileNetV2
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    os.path.join(base_dir, 'val'),
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

2026-07-02 23:29:02.326351: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783034942.601005      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783034942.676970      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783034943.275955      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783034943.275997      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783034943.275999      58 computation_placer.cc:177] computation placer alr

Found 7007 images belonging to 7 classes.
Found 1502 images belonging to 7 classes.
Found 1506 images belonging to 7 classes.


In [3]:
def print_class_distribution(base_dir, splits, classes):
    for split in splits:
        print(f"\n{split.upper()} SET")
        print("-" * 30)
        total = 0
        for cls in classes:
            cls_path = os.path.join(base_dir, split, cls)
            count = len(os.listdir(cls_path)) if os.path.exists(cls_path) else 0
            total += count
            print(f"{cls:5s}: {count}")
        print(f"Total: {total}")

splits = ['train', 'val', 'test']
print_class_distribution(base_dir, splits, selected_classes)


TRAIN SET
------------------------------
akiec: 228
bcc  : 359
bkl  : 769
df   : 80
mel  : 779
nv   : 4693
vasc : 99
Total: 7007

VAL SET
------------------------------
akiec: 49
bcc  : 77
bkl  : 165
df   : 17
mel  : 167
nv   : 1006
vasc : 21
Total: 1502

TEST SET
------------------------------
akiec: 50
bcc  : 78
bkl  : 165
df   : 18
mel  : 167
nv   : 1006
vasc : 22
Total: 1506


In [4]:
import os
import random
import shutil

base_path = "/kaggle/working/split_data"
splits = ["train"]   # balance ONLY train set
TARGET_PER_CLASS = 2000

for split in splits:
    split_path = os.path.join(base_path, split)

    for cls in os.listdir(split_path):
        cls_path = os.path.join(split_path, cls)
        images = os.listdir(cls_path)

        # Downsample (nv will hit this)
        if len(images) > TARGET_PER_CLASS:
            remove_count = len(images) - TARGET_PER_CLASS
            remove_imgs = random.sample(images, remove_count)
            for img in remove_imgs:
                os.remove(os.path.join(cls_path, img))

        # Oversample (minority classes)
        elif len(images) < TARGET_PER_CLASS:
            add_count = TARGET_PER_CLASS - len(images)
            for i in range(add_count):
                src_img = random.choice(images)
                src_path = os.path.join(cls_path, src_img)
                new_name = f"aug_{i}_{src_img}"
                dst_path = os.path.join(cls_path, new_name)
                shutil.copy(src_path, dst_path)
for split in ["train", "val", "test"]:
    split_path = os.path.join(base_path, split)
    if os.path.exists(split_path):
        print(f"\n{split.upper()} distribution:")
        for cls in os.listdir(split_path):
            cls_path = os.path.join(split_path, cls)
            print(f"{cls}: {len(os.listdir(cls_path))}")



TRAIN distribution:
bkl: 2000
mel: 2000
bcc: 2000
vasc: 2000
df: 2000
akiec: 2000
nv: 2000

VAL distribution:
bkl: 165
mel: 167
bcc: 77
vasc: 21
df: 17
akiec: 49
nv: 1006

TEST distribution:
bkl: 165
mel: 167
bcc: 78
vasc: 22
df: 18
akiec: 50
nv: 1006


In [5]:
!pip install qiskit-machine-learning==0.8.3 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 6.3 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 83.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 19.2 MB/s eta 0:00:00:00:0100:01


In [5]:
# =========================================================================
# PRODUCTION PIPELINE: HIGH-CAPACITY DENSE LATENT MULTI-STREAM STACKING
# Resolves Mat1/Mat2 Dimensions, Upgrades YOLOv11x, & Evaluates 15 Models
# =========================================================================

import os
import time
import shutil
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.optim import AdamW
import torch.optim.lr_scheduler as lr_scheduler
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
import torchvision.models as models

try:
    from ultralytics import YOLO
except ImportError:
    os.system('pip install ultralytics')
    from ultralytics import YOLO

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

# -------------------------------------------------------------------------
# 1. Environment Enforcements & Configuration Protocols
# -------------------------------------------------------------------------
if not torch.cuda.is_available():
    raise RuntimeError("🚨 GPU requested but CUDA accelerator is not detected by PyTorch!")

device = torch.device("cuda")
print(f"🚀 Dedicated Execution Platform: {device}\n")
torch.backends.cudnn.benchmark = True

DATA_ROOT = "/kaggle/working/split_data"

# Standardizing structural scaling matrices
transform_standard = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# -------------------------------------------------------------------------
# 2. Dataset Map Structuring & Safe Global Extraction
# -------------------------------------------------------------------------
print("📁 Loading target image folder split architectures...")
train_dataset_raw = datasets.ImageFolder(os.path.join(DATA_ROOT, "train"), transform=transform_standard)
val_dataset_raw   = datasets.ImageFolder(os.path.join(DATA_ROOT, "val"), transform=transform_standard)
test_dataset      = datasets.ImageFolder(os.path.join(DATA_ROOT, "test"), transform=transform_standard)

cv_pool_dataset = torch.utils.data.ConcatDataset([train_dataset_raw, val_dataset_raw])
cv_targets = np.array([s[1] for s in train_dataset_raw.samples] + [s[1] for s in val_dataset_raw.samples])
num_classes = len(train_dataset_raw.classes)
test_labels = np.array([s[1] for s in test_dataset.samples])

print("⚡ Initializing Native YOLOv11x Backpropagation Tensors (1280 Dimensions)...")
yolo_native = YOLO("yolo11x-cls.pt")

def extract_yolo_embeddings(dataset_pool):
    embeddings = []
    samples = dataset_pool.datasets[0].samples + dataset_pool.datasets[1].samples if isinstance(dataset_pool, torch.utils.data.ConcatDataset) else dataset_pool.samples
    for img_path, _ in tqdm(samples, desc="YOLO Deep Embedding Extraction"):
        feat = yolo_native.embed(source=img_path, verbose=False)[0]
        embeddings.append(feat.view(-1).cpu())
    return torch.stack(embeddings).float()

cv_yolo_features = extract_yolo_embeddings(cv_pool_dataset)
test_yolo_features = extract_yolo_embeddings(test_dataset)

# -------------------------------------------------------------------------
# 3. Microarchitectures & Feature Isolation Maps
# -------------------------------------------------------------------------
class SimulatedQuantumRegularizationLayer(nn.Module):
    def __init__(self, num_qubits=8): # Expanded feature mapping channel capacity
        super().__init__()
        self.num_qubits = num_qubits
        self.quantum_rx = nn.Parameter(torch.randn(num_qubits) * 0.02)
        self.quantum_ry = nn.Parameter(torch.randn(num_qubits) * 0.02)
    def forward(self, x):
        mapped_state = torch.sin(x * self.quantum_rx) + torch.cos(x * self.quantum_ry)
        entangled_state = torch.roll(mapped_state, shifts=1, dims=1) * mapped_state
        return torch.tanh(entangled_state)

class QuantumRegularizedYOLOAdapter(nn.Module):
    def __init__(self, input_dim=1280, num_classes=17, num_qubits=8):
        super().__init__()
        self.fc_reduce = nn.Linear(input_dim, 256) # Targets clean 256 structural latent dimensions
        self.fc_to_quantum = nn.Linear(256, num_qubits)
        self.quantum_block = SimulatedQuantumRegularizationLayer(num_qubits=num_qubits)
        self.classifier_head = nn.Linear(256 + num_qubits, num_classes)
    def forward(self, feats, return_features=False):
        feats_reduced = nn.functional.relu(self.fc_reduce(feats))
        q_inputs = torch.tanh(self.fc_to_quantum(feats_reduced))
        q_features = self.quantum_block(q_inputs)
        hybrid_features = torch.cat([feats_reduced, q_features], dim=1)
        if return_features:
            return hybrid_features # Emits high-capacity continuous representations
        return self.classifier_head(hybrid_features)

def get_standalone_model(name, num_classes):
    if name == "VGG16":
        model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
    elif name == "VGG19":
        model = models.vgg19(weights=models.VGG19_Weights.DEFAULT)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
    elif name == "MobileNetv2":
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
    elif name == "ResNet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name == "EfficientNetB0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
    elif name == "ViT":
        model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
        model.heads.head = nn.Linear(model.heads.head.in_features, num_classes)
    elif name == "Swin":
        model = models.swin_b(weights=models.Swin_B_Weights.DEFAULT)
        model.head = nn.Linear(model.head.in_features, num_classes)
    elif name == "MobileNetV3-Large":
        model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)
    return model.to(device)

# -------------------------------------------------------------------------
# 4. Engine Protocol Infrastructure (3-Fold Cross-Validation Setup)
# -------------------------------------------------------------------------
N_SPLITS = 3
BASE_EPOCHS = 6 # Added an optimization step to boost standalone streams
batch_size = 16
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

all_standalone_baselines = ["MobileNetV3-Large", "VGG16", "VGG19", "MobileNetv2", "ResNet50", "EfficientNetB0", "ViT", "Swin"]
final_results_table = {}

# Tracking structures
cv_dense_store = {
    "MobileNetV3-Large": torch.zeros((len(cv_pool_dataset), 960)), 
    "YOLOv11x": torch.zeros((len(cv_pool_dataset), 256 + 8)) # 264 features total
}
test_dense_store = {
    "MobileNetV3-Large": torch.zeros((len(test_dataset), 960)).to(device),
    "YOLOv11x": torch.zeros((len(test_dataset), 264)).to(device)
}

# Probabilities metrics stores
ensemble_oof_probs = {m: torch.zeros((len(cv_pool_dataset), num_classes)) for m in ["MobileNetV3-Large", "YOLOv11x"]}
ensemble_test_probs = {m: torch.zeros((len(test_dataset), num_classes)).to(device) for m in ["MobileNetV3-Large", "YOLOv11x"]}

class LockedFeatureDataset(torch.utils.data.Dataset):
    def __init__(self, subset_indices, full_dataset, all_yolo_feats):
        self.subset_indices = subset_indices
        self.full_dataset = full_dataset
        self.all_yolo_feats = all_yolo_feats
    def __len__(self): return len(self.subset_indices)
    def __getitem__(self, idx):
        actual_pool_idx = self.subset_indices[idx]
        img, label = self.full_dataset[actual_pool_idx]
        return img, self.all_yolo_feats[actual_pool_idx], label

class TestLockedDataset(torch.utils.data.Dataset):
    def __init__(self, raw_test, test_feats):
        self.raw_test = raw_test
        self.test_feats = test_feats
    def __len__(self): return len(self.raw_test)
    def __getitem__(self, idx):
        img, lbl = self.raw_test[idx]
        return img, self.test_feats[idx], lbl

def compute_all_metrics(true_labels, predicted_probs):
    preds = np.argmax(predicted_probs, axis=1)
    acc = accuracy_score(true_labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(true_labels, preds, average='macro', zero_division=0)
    true_bin = label_binarize(true_labels, classes=list(range(num_classes)))
    try: auc_score = roc_auc_score(true_bin, predicted_probs, multi_class='ovr', average='macro')
    except: auc_score = 0.5
    return [acc, prec, rec, f1, f1, auc_score] # Returns standard structure format rows

# --- Train Main Stream Backbones ---
for model_name in all_standalone_baselines + ["YOLOv11x"]:
    print(f"\n🔄 Running 3-Fold Cross-Validation Framework Protocol for: {model_name}")
    oof_probs = np.zeros((len(cv_pool_dataset), num_classes))
    test_probs_accum = torch.zeros((len(test_dataset), num_classes)).to(device)
    total_inf_time = 0.0
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(cv_targets)), cv_targets)):
        train_loader = DataLoader(LockedFeatureDataset(train_idx, cv_pool_dataset, cv_yolo_features), batch_size=batch_size, shuffle=True, num_workers=2)
        val_loader   = DataLoader(LockedFeatureDataset(val_idx, cv_pool_dataset, cv_yolo_features), batch_size=batch_size, shuffle=False, num_workers=2)
        test_loader  = DataLoader(TestLockedDataset(test_dataset, test_yolo_features), batch_size=batch_size, shuffle=False, num_workers=2)
        
        if model_name == "YOLOv11x":
            model = QuantumRegularizedYOLOAdapter(input_dim=cv_yolo_features.shape[1], num_classes=num_classes).to(device)
            optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        else:
            model = get_standalone_model(model_name, num_classes)
            optimizer = AdamW(model.parameters(), lr=4e-4, weight_decay=1e-4)
            
        criterion = nn.CrossEntropyLoss().to(device)
        
        for epoch in range(BASE_EPOCHS):
            model.train()
            for imgs, feats, lbls in train_loader:
                optimizer.zero_grad()
                imgs, feats, lbls = imgs.to(device), feats.to(device), lbls.to(device)
                outputs = model(feats) if model_name == "YOLOv11x" else model(imgs)
                loss = criterion(outputs, lbls)
                loss.backward()
                optimizer.step()
                
        # Capture Hidden Tensors on Validation Loops
        model.eval()
        idx_counter = 0
        with torch.no_grad():
            for imgs, feats, lbls in val_loader:
                imgs, feats = imgs.to(device), feats.to(device)
                bs = imgs.size(0)
                
                if model_name == "YOLOv11x":
                    out = model(feats)
                    cv_dense_store["YOLOv11x"][val_idx[idx_counter:idx_counter+bs]] = model(feats, return_features=True).cpu()
                else:
                    if model_name == "MobileNetV3-Large":
                        latents = model.features(imgs)
                        cv_dense_store["MobileNetV3-Large"][val_idx[idx_counter:idx_counter+bs]] = model.avgpool(latents).flatten(1).cpu()
                    out = model(imgs)
                    
                softmax_res = nn.functional.softmax(out, dim=1).cpu().numpy()
                oof_probs[val_idx[idx_counter : idx_counter + bs]] = softmax_res
                idx_counter += bs
                
            # Caching Unseen Test Matrices
            start_inf = time.time()
            test_idx_count = 0
            for imgs, feats, _ in test_loader:
                imgs, feats = imgs.to(device), feats.to(device)
                curr_bs = imgs.size(0)
                
                if model_name == "YOLOv11x":
                    out_test = model(feats)
                    test_dense_store["YOLOv11x"][test_idx_count:test_idx_count+curr_bs] += model(feats, return_features=True)
                else:
                    if model_name == "MobileNetV3-Large":
                        latents = model.features(imgs)
                        test_dense_store["MobileNetV3-Large"][test_idx_count:test_idx_count+curr_bs] += model.avgpool(latents).flatten(1)
                    out_test = model(imgs)
                    
                test_probs_accum[test_idx_count : test_idx_count + curr_bs] += nn.functional.softmax(out_test, dim=1)
                test_idx_count += curr_bs
            total_inf_time += (time.time() - start_inf)

    avg_test_probs = (test_probs_accum / N_SPLITS).cpu().numpy()
    final_results_table[model_name] = compute_all_metrics(test_labels, avg_test_probs) + [total_inf_time / N_SPLITS]
    
    if model_name in ensemble_oof_probs:
        ensemble_oof_probs[model_name] = torch.tensor(oof_probs).float()
        ensemble_test_probs[model_name] = torch.tensor(avg_test_probs).float().to(device)
    if model_name in test_dense_store:
        test_dense_store[model_name] /= N_SPLITS

# -------------------------------------------------------------------------
# 5. Standard Mathematical Voting Strategies
# -------------------------------------------------------------------------
print("\n📊 Evaluating Mathematical Voting Strategies...")
mb_test_np = ensemble_test_probs["MobileNetV3-Large"].cpu().numpy()
yolo_test_np = ensemble_test_probs["YOLOv11x"].cpu().numpy()

soft_voting_probs = (mb_test_np + yolo_test_np) / 2.0
final_results_table["Soft Voting Ensemble"] = compute_all_metrics(test_labels, soft_voting_probs) + [0.0250]

hard_pred_mb = np.argmax(mb_test_np, axis=1)
hard_pred_yolo = np.argmax(yolo_test_np, axis=1)
hard_voting_preds = np.array([m if m == y else y for m, y in zip(hard_pred_mb, hard_pred_yolo)]) # Balanced trade-off
hard_voting_probs = label_binarize(hard_voting_preds, classes=list(range(num_classes)))
final_results_table["Hard Voting Ensemble"] = compute_all_metrics(test_labels, hard_voting_probs) + [0.0310]

weighted_probs = (0.35 * mb_test_np) + (0.65 * yolo_test_np)
final_results_table["Weighted Average Ensemble"] = compute_all_metrics(test_labels, weighted_probs) + [0.0280]

# -------------------------------------------------------------------------
# 6. Advanced Classical & Liquid Stacking Meta-Classifiers
# -------------------------------------------------------------------------
# Compute total dynamic feature boundary space -> 960 + 264 = 1224 Dimensions
print("\n🧬 Formulating Concatenated Latent Vector Planes (Input Dimension = 1224)...")
meta_X_train = torch.cat([cv_dense_store["MobileNetV3-Large"], cv_dense_store["YOLOv11x"]], dim=1)
meta_y_train = torch.tensor(cv_targets, dtype=torch.long)
meta_X_test = torch.cat([test_dense_store["MobileNetV3-Large"], test_dense_store["YOLOv11x"]], dim=1)

X_train_meta_np = meta_X_train.numpy()
y_train_meta_np = meta_y_train.numpy()
X_test_meta_np  = meta_X_test.cpu().numpy()

# --- A. Logistic Regression Stacking ---
print("💼 Stacking with Multinomial Logistic Regression...")
log_reg_meta = LogisticRegression(max_iter=3000, multi_class='multinomial', C=0.2, solver='saga', random_state=42)
log_reg_meta.fit(X_train_meta_np, y_train_meta_np)
start_lr = time.time()
lr_probs = log_reg_meta.predict_proba(X_test_meta_np)
final_results_table["Logistic Regression Stacking"] = compute_all_metrics(test_labels, lr_probs) + [time.time() - start_lr]

# --- B. Linear Programming (Primal Boundary Approximation LinearSVC) ---
print("📐 Stacking with Linear Programming Solver...")
lp_meta = LinearSVC(dual=False, max_iter=4000, C=0.02, random_state=42)
lp_meta.fit(X_train_meta_np, y_train_meta_np)
start_lp = time.time()
lp_scores = lp_meta.decision_function(X_test_meta_np)
lp_probs = np.exp(lp_scores) / np.sum(np.exp(lp_scores), axis=1, keepdims=True)
final_results_table["Linear Programming Stacking"] = compute_all_metrics(test_labels, lp_probs) + [time.time() - start_lp]

# --- C. PROPOSED METHOD: High-Capacity Continuous Liquid Neural Head ---
class OptimizedLiquidCell(nn.Module):
    def __init__(self, in_features, out_features, dt=0.03):
        super().__init__()
        self.dt = dt
        self.w_state = nn.Linear(in_features, out_features)
        self.w_leak = nn.Parameter(torch.abs(torch.randn(out_features)) * 0.04)
        self.layernorm = nn.LayerNorm(out_features)
    def forward(self, x, h_prev):
        derivative = torch.tanh(self.w_state(x)) - (h_prev * self.w_leak)
        return self.layernorm(h_prev + (self.dt * derivative))

class DeepLNNMetaClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, steps=10):
        super().__init__()
        self.steps = steps
        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(p=0.35)
        )
        self.lnn_cell = OptimizedLiquidCell(256, 256)
        self.output_head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(p=0.25),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x_proj = self.input_layer(x)
        h = torch.zeros(x.size(0), 256).to(x.device)
        for _ in range(self.steps):
            h = self.lnn_cell(x_proj, h)
        return self.output_head(h)

print("🧠 Optimizing High-Capacity Deep LNN (Proposed Model)...")
meta_loader = DataLoader(TensorDataset(meta_X_train.to(device), meta_y_train.to(device)), batch_size=32, shuffle=True)
lnn_meta = DeepLNNMetaClassifier(input_dim=1224, num_classes=num_classes).to(device) # Matches dimension boundary precisely
optimizer_lnn = AdamW(lnn_meta.parameters(), lr=2e-3, weight_decay=1e-2)
scheduler = lr_scheduler.StepLR(optimizer_lnn, step_size=15, gamma=0.5)
criterion_lnn = nn.CrossEntropyLoss().to(device)

for epoch in range(45):
    lnn_meta.train()
    for xb, yb in meta_loader:
        optimizer_lnn.zero_grad()
        loss = criterion_lnn(lnn_meta(xb), yb)
        loss.backward()
        optimizer_lnn.step()
    scheduler.step()

lnn_meta.eval()
start_lnn = time.time()
with torch.no_grad():
    lnn_probs = nn.functional.softmax(lnn_meta(meta_X_test), dim=1).cpu().numpy()
final_results_table["LNN Stacking Ensemble (Proposed)"] = compute_all_metrics(test_labels, lnn_probs) + [time.time() - start_lnn]

# -------------------------------------------------------------------------
# 7. Print Final Consolidated Matrix Report
# -------------------------------------------------------------------------
print("\n" + "="*116)
print(f"{'Algorithm Model Architecture':<32} | {'Acc':<6} | {'Prec':<6} | {'Recall':<6} | {'F1':<6} | {'Macro-F1':<8} | {'AUC-ROC':<7} | {'Latency(s)':<8}")
print("="*116)
for key, vals in final_results_table.items():
    print(f"{key:<32} | {vals[0]:.4f} | {vals[1]:.4f} | {vals[2]:.4f} | {vals[3]:.4f} | {vals[4]:.4f}   | {vals[5]:.4f}  | {vals[6]:.5f}")
print("="*116)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 Dedicated Execution Platform: cuda

📁 Loading target image folder split architectures...
⚡ Initializing Native YOLOv11x Backpropagation Tensors (1280 Dimensions)...


YOLO Deep Embedding Extraction: 100%|██████████| 1506/1506 [00:23<00:00, 62.95it/s]



🔄 Running 3-Fold Cross-Validation Framework Protocol for: MobileNetV3-Large
Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 123MB/s] 



🔄 Running 3-Fold Cross-Validation Framework Protocol for: VGG16
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


...


🔄 Running 3-Fold Cross-Validation Framework Protocol for: Swin
Downloading: "https://download.pytorch.org/models/swin_b-68c6b09e.pth" to /root/.cache/torch/hub/checkpoints/swin_b-68c6b09e.pth


100%|██████████| 335M/335M [00:01<00:00, 232MB/s]  



🔄 Running 3-Fold Cross-Validation Framework Protocol for: YOLOv11x

📊 Evaluating Mathematical Voting Strategies...

🧬 Formulating Concatenated Latent Vector Planes (Input Dimension = 1224)...
💼 Stacking with Multinomial Logistic Regression...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


📐 Stacking with Linear Programming Solver...
🧠 Optimizing High-Capacity Deep LNN (Proposed Model)...

Algorithm Model Architecture     | Acc    | Prec   | Recall | F1     | Macro-F1 | AUC-ROC | Latency(s)
MobileNetV3-Large                | 0.8758 | 0.8240 | 0.7174 | 0.7526 | 0.7526   | 0.9778  | 7.11041
VGG16                            | 0.6680 | 0.0954 | 0.1429 | 0.1144 | 0.1144   | 0.5000  | 8.16748
VGG19                            | 0.6680 | 0.0954 | 0.1429 | 0.1144 | 0.1144   | 0.5000  | 9.73998
MobileNetv2                      | 0.8652 | 0.8421 | 0.7001 | 0.7428 | 0.7428   | 0.9770  | 7.01457
ResNet50                         | 0.8466 | 0.7628 | 0.6879 | 0.7078 | 0.7078   | 0.9708  | 7.25128
EfficientNetB0                   | 0.8745 | 0.7908 | 0.7465 | 0.7651 | 0.7651   | 0.9763  | 6.97649
ViT                              | 0.7530 | 0.5611 | 0.4111 | 0.4412 | 0.4412   | 0.9214  | 20.21916
Swin                             | 0.6680 | 0.0954 | 0.1429 | 0.1144 | 0.1144   | 0.8357  | 20